# Lesson 1 — Lakehouse Basics: Unity Catalog namespace, notebooks, and your first Delta tables

**Exam objectives** (Data Engineer Associate, May 2026 outline; also the foundation for Professional):

* Databricks Intelligence Platform: workspace, compute types, notebooks, Unity Catalog object model.
* Data Transformation and Modeling: create and modify Delta tables with SQL and PySpark.
* Delta Lake: transaction log, `DESCRIBE HISTORY`, time travel, `RESTORE`.

**5 Learning Objectives**

1. **Use** the Unity Catalog three-level namespace (`catalog.schema.table`) to navigate and address tables.
2. **Explain** what a Delta table is (Parquet + `_delta_log`) and how the transaction log enables ACID, history, and time travel.
3. **Compare** `INSERT OVERWRITE`, `CREATE OR REPLACE TABLE`, and `DROP`+`CREATE` by what each does to table history.
4. **Implement** DML (`CREATE TABLE AS SELECT`, `INSERT`, `UPDATE`, `DELETE`, `MERGE INTO`) in both SQL and PySpark.
5. **Diagnose** table changes using `DESCRIBE HISTORY`, time travel (`VERSION AS OF` / `TIMESTAMP AS OF`), and `RESTORE TABLE`.

> **Evidence-only rule:** Every number in a markdown cell comes from a cell that was actually run in this notebook. If something could not be verified on serverless compute, it is marked **NOT VERIFIED ON SERVERLESS — concept only**.

In [0]:
# Classroom Setup — idempotent (DROP schema CASCADE, then recreate).
# Fixed random seed so all counts are reproducible.
import random, datetime
from pyspark.sql import Row

rng = random.Random(42)  # isolated generator: Spark calls cannot consume its state, so all counts are deterministic

# Idempotent: drop everything, start clean
spark.sql("DROP SCHEMA IF EXISTS main.dbacademy_l01 CASCADE")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.dbacademy_l01")
spark.sql("CREATE VOLUME IF NOT EXISTS main.dbacademy_l01.data")

# --- name pools for fake (no PHI) healthcare data ---
first_names = ["James","Mary","John","Patricia","Robert","Jennifer","Michael","Linda",
               "William","Elizabeth","David","Barbara","Richard","Susan","Joseph","Jessica"]
last_names  = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis",
               "Rodriguez","Martinez","Hernandez","Lopez","Gonzalez","Wilson","Anderson","Thomas"]
cities = [("Boston","MA"),("Springfield","MA"),("Hartford","CT"),("Providence","RI"),("Portland","ME")]
genders = ["F","M"]
enc_types = ["OUTPATIENT","INPATIENT","EMERGENCY","TELEHEALTH"]
diag_codes = ["Z00.00","E11.9","I10","J06.9","M54.5","R51.9","K21.9","N39.0"]
specialties = ["Cardiology","Neurology","Oncology","Pediatrics","Orthopedics","Radiology","General","Dermatology"]
hospitals = ["General Hospital","St Mary Medical","Lakeside Health","Riverside Clinic"]
claim_statuses = ["SUBMITTED","APPROVED","DENIED","PENDING"]
base_date = datetime.date(2024, 1, 1)

# patients: 50 rows
patients = [Row(patient_id=i,
                 patient_name=f"{rng.choice(first_names)} {rng.choice(last_names)}",
                 dob=(base_date - datetime.timedelta(days=rng.randint(7000, 30000))).isoformat(),
                 gender=rng.choice(genders),
                 city=rng.choice(cities)[0], state=rng.choice(cities)[1],
                 created_at=(base_date + datetime.timedelta(days=rng.randint(0, 400))).isoformat())
            for i in range(1, 51)]
spark.createDataFrame(patients).write.mode("overwrite").saveAsTable("main.dbacademy_l01.patients")

# providers: 10 rows
providers = [Row(provider_id=i,
                 provider_name=f"Dr {rng.choice(first_names)} {rng.choice(last_names)}",
                 specialty=rng.choice(specialties), npi=1000000000 + i,
                 hospital=rng.choice(hospitals)) for i in range(1, 11)]
spark.createDataFrame(providers).write.mode("overwrite").saveAsTable("main.dbacademy_l01.providers")

# encounters: 300 rows
encounters = [Row(encounter_id=i, patient_id=rng.randint(1, 50), provider_id=rng.randint(1, 10),
                  encounter_date=(base_date + datetime.timedelta(days=rng.randint(0, 365))).isoformat(),
                  encounter_type=rng.choice(enc_types), diagnosis_code=rng.choice(diag_codes),
                  created_at=(base_date + datetime.timedelta(days=rng.randint(0, 400))).isoformat())
             for i in range(1, 301)]
spark.createDataFrame(encounters).write.mode("overwrite").saveAsTable("main.dbacademy_l01.encounters")

# claims: 200 rows (some will be updated/deleted later in the lesson)
claims = [Row(claim_id=i, encounter_id=rng.randint(1, 300), patient_id=rng.randint(1, 50),
              provider_id=rng.randint(1, 10),
              claim_date=(base_date + datetime.timedelta(days=rng.randint(0, 365))).isoformat(),
              claim_amount=round(rng.uniform(50, 5000), 2),
              claim_status=rng.choice(claim_statuses),
              submitted_at=(base_date + datetime.timedelta(days=rng.randint(0, 400))).isoformat())
          for i in range(1, 201)]
spark.createDataFrame(claims).write.mode("overwrite").saveAsTable("main.dbacademy_l01.claims")

# Write a small CSV into the volume (used by Example E2)
dbutils.fs.put("/Volumes/main/dbacademy_l01/data/providers_sample.csv",
               "provider_id,provider_name,specialty\n1,Dr Test,Cardiology\n2,Dr Demo,Neurology\n", overwrite=True)

# --- verify real counts with asserts ---
expected = {"patients": 50, "providers": 10, "encounters": 300, "claims": 200}
for t, n in expected.items():
    cnt = spark.table(f"main.dbacademy_l01.{t}").count()
    assert cnt == n, f"{t}: expected {n}, got {cnt}"
    print(f"{t}: {cnt}")
print("SETUP OK — schema main.dbacademy_l01 ready")

## 3.1 Concept — Unity Catalog three-level namespace

**What it is.** Unity Catalog (UC) is the governance layer for all data in a Databricks workspace. Every table, view, volume, function, and model lives in a three-level namespace: `catalog.schema.table` (or `catalog.schema.volume`).

**Why it exists.** Before UC, tables lived in a flat `database.table` space with no central access control. UC adds a top level (`catalog`) so teams can isolate data, enforce one set of grants, and track lineage across the whole workspace.

**When to use vs the alternative.**

| Need | Use | Instead of |
| --- | --- | --- |
| Address a table unambiguously | `main.dbacademy_l01.claims` (fully qualified) | bare `claims` (depends on current catalog/schema) |
| Set a default so you can use short names | `USE CATALOG main; USE SCHEMA dbacademy_l01` | typing the full name every time |
| Store files (not tables) | UC **volume** `/Volumes/catalog/schema/volume` | DBFS root `/` (deprecated, no governance) |

**Minimal syntax.**

```sql
SHOW CATALOGS;                         -- list catalogs you can see
USE CATALOG main;                      -- set default catalog
SHOW SCHEMAS IN main;                  -- list schemas in that catalog
USE SCHEMA dbacademy_l01;              -- set default schema
SHOW TABLES;                           -- list tables in current schema
SELECT current_catalog(), current_schema();   -- check defaults
```

> **Verified in this notebook:** the default catalog before any `USE` is `bunyod` (your workspace default), **not** `main`. You must `USE CATALOG main` before `current_catalog()` returns `main`.

> **Exam trap:** `current_schema()` returns only the schema name (e.g. `dbacademy_l01`), not `catalog.schema`. Use `current_catalog()` + `current_schema()` together.

## 3.2 Concept — Notebook essentials

**What it is.** A Databricks notebook is a list of **cells**. Each cell has a language chosen by a magic at the top of the cell: `%sql`, `%python` (default on this compute), `%md` (markdown), `%run`, `%fs` (dbutils.fs), `%sh` (shell).

**Why it exists.** One notebook can mix SQL, Python, and prose so you can explore, transform, and document in one place.

**When to use vs the alternative.**

| Want | Use | Note |
| --- | --- | --- |
| Run SQL | `%sql SELECT ...` | one statement or a few |
| Run Python / PySpark | `%python` or plain (default here) | full Python, `spark`, `dbutils` available |
| Parameterize a notebook | `dbutils.widgets` | pass values between runs / jobs |
| Run another notebook inline | `%run /path/to/notebook` | shares the caller's context |
| Run a notebook and get a return value | `dbutils.notebook.run(path, timeout, args)` | returns a string |
| List files in a volume | `dbutils.fs.ls("/Volumes/...")` | NOT `sh` `ls` |

**Minimal syntax.**

```python
# spark.table vs spark.sql
spark.table("main.dbacademy_l01.claims")      # returns a DataFrame
spark.sql("SELECT * FROM main.dbacademy_l01.claims")  # also a DataFrame

# display() vs show()
display(df)   # rich HTML table + chart button (notebook UI)
df.show()     # plain text table (works anywhere)

# SQL <-> DataFrame round-trip
df.createOrReplaceTempView("tv")        # DataFrame -> SQL name (session-scoped)
spark.sql("SELECT * FROM tv")           # query it back as SQL
```

**Temp view vs global temp view vs table** (deep dive in Example E8):

| Object | Scope | Created by | Where visible |
| --- | --- | --- | --- |
| Temp view | current Spark session | `createOrReplaceTempView` | this notebook session only |
| Global temp view | across sessions | `createGlobalTempView` | `global_temp.` prefix — **NOT VERIFIED ON SERVERLESS: fails with NOT_SUPPORTED_WITH_SERVERLESS** |
| UC table / view | persistent, governed | `CREATE TABLE` / `saveAsTable` | everyone with grants, survives restart |

> **Verified in this notebook:** global temp views are **not supported on serverless compute**. Use a UC table or a session temp view instead.

## 3.3 Concept — Compute: what runs where

**What it is.** Databricks has three main compute types:

| Compute | Typical use | What runs notebooks? |
| --- | --- | --- |
| **Serverless** all-purpose | interactive notebooks, fast start, no cluster to manage | yes (this notebook) |
| **Classic** all-purpose | interactive, long-lived, custom clusters / libraries | yes |
| **Lakeflow Jobs** compute | scheduled / triggered pipelines, autoscaling, cheaper | runs jobs, not interactive |
| **SQL warehouse** | BI dashboards, SQL queries, Genie | SQL only (no notebooks) |

**Why it exists.** Different workloads need different price/performance. Serverless starts in seconds and scales per-query. Jobs compute is cheaper for batch. SQL warehouses serve BI.

**When to use vs the alternative.**

* Interactive exploring / prototyping → serverless all-purpose.
* Scheduled ETL → Lakeflow Jobs compute (or a serverless job).
* Dashboards & ad-hoc SQL → SQL warehouse.

> **NOT VERIFIED ON SERVERLESS — concept only:** classic all-purpose and Jobs compute details (cluster policies, autoscaling ranges, DBR versions) are not exercised on this serverless notebook.

## 3.4 Concept — Delta Lake: what a Delta table is

**What it is.** A Delta table is a folder of **Parquet data files** plus a transaction log folder `_delta_log/`. Every change (insert, update, delete, optimize) appends a JSON **commit** to the log. The log is the single source of truth for what the table contains right now.

**Why it exists.** Plain Parquet has no transaction log, so concurrent writes can corrupt data and you cannot update/delete rows in place. Delta adds ACID transactions, schema enforcement, and time travel on top of Parquet.

**When to use vs the alternative.**

| Need | Use | Instead of |
| --- | --- | --- |
| Reliable read/write with ACID | Delta table | plain Parquet |
| Update / delete rows | `UPDATE`/`DELETE`/`MERGE` on Delta | rewriting Parquet files by hand |
| Look at old versions of the data | time travel (`VERSION AS OF`) | impossible on plain Parquet |

**Managed vs external table** (concept only here; deep dive in Lesson 2):

* **Managed table:** UC owns the storage location. `DROP TABLE` deletes the data.
* **External table:** you register an existing data path. `DROP TABLE` removes only the metadata; files stay.

> **NOT VERIFIED ON SERVERLESS — concept only:** external table `LOCATION` behavior and `DROP` file retention are covered in Lesson 2.

**Minimal syntax.**

```sql
CREATE TABLE main.dbacademy_l01.my_table (id INT, val STRING) USING DELTA;
-- on UC, Delta is the default format, so USING DELTA is optional
```

## 3.5 Concept — DML: create, insert, update, delete, merge

**What it is.** Delta supports standard SQL DML plus `MERGE INTO`, and the same operations in PySpark.

**Why it exists.** You need to create tables from queries, append rows, and modify or remove rows — all with ACID guarantees and a logged history.

**When to use vs the alternative.**

| Goal | SQL | PySpark |
| --- | --- | --- |
| Create table from a query | `CREATE TABLE t AS SELECT ...` (CTAS) | `df.write.saveAsTable("t")` |
| Overwrite all rows, keep history | `INSERT OVERWRITE TABLE t SELECT ...` | `df.write.mode("overwrite").saveAsTable(...)` |
| Append rows | `INSERT INTO t VALUES ...` | `df.write.mode("append").saveAsTable(...)` |
| Update rows | `UPDATE t SET ... WHERE ...` | `DeltaTable.forName(spark,"t").update(...)` |
| Delete rows | `DELETE FROM t WHERE ...` | `DeltaTable.forName(spark,"t").delete(...)` |
| Upsert (insert + update + delete) | `MERGE INTO ... WHEN MATCHED / NOT MATCHED / NOT MATCHED BY SOURCE` | `DeltaTable.forName(spark,"t").merge(...)` |

**Minimal syntax — MERGE (the three branches).**

```sql
MERGE INTO target AS t USING source AS s ON t.id = s.id
WHEN MATCHED [AND <cond>] THEN UPDATE SET t.col = s.col
WHEN NOT MATCHED [AND <cond>] THEN INSERT (cols) VALUES (s.cols)
WHEN NOT MATCHED BY SOURCE [AND <cond>] THEN DELETE   -- or UPDATE
```

* `WHEN MATCHED` — a target row that joins to a source row.
* `WHEN NOT MATCHED` — a source row with no target match → **insert**.
* `WHEN NOT MATCHED BY SOURCE` — a target row with no source match → **delete** (or update).

> **Exam trap:** `WHEN NOT MATCHED` inserts from the **source**. `WHEN NOT MATCHED BY SOURCE` acts on the **target**. They are opposites — easy to mix up.

## 3.6 Concept — Table history and time travel

**What it is.** Every commit to a Delta table is a numbered **version** (starting at 0) recorded in the log. `DESCRIBE HISTORY` lists all versions with the operation name, parameters, and metrics. Time travel lets you query the table **as it was at a past version or timestamp**.

**Why it exists.** The log gives you an audit trail and the ability to undo mistakes: you can read old data and `RESTORE` the table to a known-good version.

**When to use vs the alternative.**

| Goal | Use |
| --- | --- |
| See the audit trail | `DESCRIBE HISTORY t` |
| See physical table info (files, size, location) | `DESCRIBE DETAIL t` |
| See schema + extended metadata | `DESCRIBE EXTENDED t` |
| Read an old version | `SELECT ... FROM t VERSION AS OF 1` |
| Read at a past timestamp | `SELECT ... FROM t TIMESTAMP AS OF '2024-06-01'` |
| Roll back the table | `RESTORE TABLE t TO VERSION AS OF 1` |

**Minimal syntax.**

```sql
DESCRIBE HISTORY main.dbacademy_l01.claims;                         -- audit trail
SELECT COUNT(*) FROM main.dbacademy_l01.claims VERSION AS OF 0;     -- old version
RESTORE TABLE main.dbacademy_l01.claims TO VERSION AS OF 0;          -- roll back
```

> **Exam trap:** the RESTORE syntax is `RESTORE TABLE t TO VERSION AS OF n` (with `AS OF`). Writing `TO VERSION n` is a syntax error — verified in this notebook.

> **Verified in this notebook:** `RESTORE` does **not** erase history — it creates a **new** version whose operation is `RESTORE`. Old versions are still readable.

## 4 Examples

Each example follows the same pattern: **Predict** (answer before running) → **code** (every line commented) → **Explained** (the real output numbers from this run).

---

### Example E1 — Namespace navigation and SHOW commands

**Predict (answer before you run the next cell):**
1. What does `current_catalog()` return *before* any `USE` statement?
2. After `USE SCHEMA dbacademy_l01`, can you query `patients` without the `main.dbacademy_l01.` prefix?
3. How many columns does `SHOW COLUMNS IN patients` return?

In [0]:
# E1: navigate the three-level namespace with SHOW commands
# display() renders each result so every output the Explained cell mentions is visible.
display(spark.sql("SELECT current_catalog() AS default_catalog"))   # before any USE: the workspace default catalog
spark.sql("USE CATALOG main")                  # set the default catalog to main (no result to display)
spark.sql("USE SCHEMA dbacademy_l01")          # set the default schema
display(spark.sql("SELECT current_catalog() AS catalog_now, current_schema() AS schema_now"))  # confirm defaults
display(spark.sql("SHOW TABLES"))               # list tables in the current schema (unqualified)
display(spark.sql("SELECT COUNT(*) AS patients_unqualified_count FROM patients"))  # unqualified name works after USE SCHEMA
display(spark.sql("SHOW COLUMNS IN patients")) # list columns of the patients table

**Explained (real output from this run):**

* `current_catalog()` before any `USE` returned **`bunyod`** — the workspace default, *not* `main`. This is why you should always `USE CATALOG` explicitly.
* After `USE CATALOG main` + `USE SCHEMA dbacademy_l01`, `current_catalog()` = **`main`** and `current_schema()` = **`dbacademy_l01`**.
* `SHOW TABLES` returned **4 tables**: `claims`, `encounters`, `patients`, `providers`.
* The unqualified query `SELECT COUNT(*) FROM patients` worked and returned **50** — `USE SCHEMA` set the default, so the two-level prefix is optional.
* `SHOW COLUMNS IN patients` returned **7 columns**: `patient_id, patient_name, dob, gender, city, state, created_at`.

> **Key takeaway:** the fully qualified name `main.dbacademy_l01.patients` always works. Short names work only after `USE CATALOG` + `USE SCHEMA` set the right defaults.

### Example E2 — Widgets + dbutils.fs on a volume

**Predict:**
1. What does `dbutils.fs.ls` return for the volume path — a list of what?
2. After creating a text widget, how do you read its value back in SQL/Python?

> `%run` is a **concept** here: it needs a second notebook file to run, so we do not execute it. `%run /path/to/nb` runs that notebook inline and shares its variables. `dbutils.notebook.run(path, timeout, args)` runs it as a sub-call and returns a string.

In [0]:
# E2: dbutils.fs on a UC volume + dbutils.widgets
# The setup cell wrote a small CSV into the volume; list it with dbutils.fs.ls
print("=== dbutils.fs.ls on the volume ===")
for f in dbutils.fs.ls("/Volumes/main/dbacademy_l01/data"):
    print(f"  path={f.path}  size={f.size}  isDir={f.isDir()}")  # f.isDir() is a method call

print("=== read the file head ===")
print(dbutils.fs.head("/Volumes/main/dbacademy_l01/data/providers_sample.csv"))

# dbutils.widgets: create a parameter, read it, use it, then remove it
print("=== widgets ===")
dbutils.widgets.text("provider_specialty", "Cardiology", "Provider Specialty")  # name, default, label
wval = dbutils.widgets.get("provider_specialty")  # read the current widget value
print("widget value:", wval)
cnt = spark.sql(f"SELECT COUNT(*) AS c FROM providers WHERE specialty = '{wval}'").collect()[0]["c"]
print(f"providers with specialty={wval}:", cnt)
dbutils.widgets.remove("provider_specialty")  # clean up the widget
print("widget removed")

**Explained (real output from this run):**

* `dbutils.fs.ls` returned one file: `providers_sample.csv`, **size 77 bytes**, `isDir=False`. Use `f.isDir()` (method call) to test directories.
* `dbutils.fs.head` printed the CSV content (header + 2 rows).
* The widget `provider_specialty` defaulted to **`Cardiology`**; querying providers with that specialty returned **4**.
* `dbutils.widgets.remove("provider_specialty")` cleared the widget.

> **Exam point:** `dbutils.fs` works with UC volume paths (`/Volumes/...`). It is the preferred file API; avoid `sh` `ls`/`cat`.

### Example E3 — CTAS + INSERT + DESCRIBE HISTORY

**Predict:**
1. A `CREATE OR REPLACE TABLE ... AS SELECT` creates which **version** number in history?
2. After an `INSERT INTO` (append), what is the `operation` name and the `mode` parameter?
3. How many versions are in the history after one CTAS + one INSERT?

In [0]:
# E3: CREATE TABLE AS SELECT, then INSERT, then read the history
# CTAS creates a new table from a query (its first version).
spark.sql("""
CREATE OR REPLACE TABLE main.dbacademy_l01.ex_denied_claims AS
SELECT claim_id, patient_id, claim_amount, claim_status
FROM main.dbacademy_l01.claims
WHERE claim_status = 'DENIED'
""")
display(spark.sql("SELECT COUNT(*) AS after_ctas_count FROM main.dbacademy_l01.ex_denied_claims"))  # count right after CTAS

# INSERT INTO appends rows (a new WRITE version, mode = Append)
spark.sql("""
INSERT INTO main.dbacademy_l01.ex_denied_claims
VALUES (9991, 1, 250.00, 'DENIED'), (9992, 2, 500.00, 'DENIED')
""")
display(spark.sql("SELECT COUNT(*) AS after_insert_count FROM main.dbacademy_l01.ex_denied_claims"))  # count after insert

display(spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_denied_claims"))  # show version, operation, metrics

**Explained (real output from this run):**

* CTAS produced **40 rows** (the number of `DENIED` claims out of 200).
* After `INSERT INTO` (2 rows), the count became **42**.
* `DESCRIBE HISTORY` showed **2 versions**:
  * **version 0** — operation `CREATE OR REPLACE TABLE AS SELECT`, `numOutputRows = 40`.
  * **version 1** — operation `WRITE`, parameters `mode -> Append`, `numOutputRows = 2`.

> **Key takeaway:** CTAS is version 0. Each append `INSERT` adds a `WRITE` version with `mode = Append`. History grows by one version per commit.
>
> **Real-world note (verified):** on this serverless workspace, **predictive optimization** may insert extra `OPTIMIZE` versions you did not write. Your version numbers can be higher than expected — always read the `operation` column, not just the number.

### Example E4 — UPDATE / DELETE + time travel

**Predict:**
1. After `UPDATE ... SET encounter_type='VIDEO_VISIT' WHERE encounter_type='TELEHEALTH'`, how many `TELEHEALTH` rows remain?
2. After `DELETE WHERE encounter_type='EMERGENCY'`, what is the total row count?
3. If you query the table `VERSION AS OF 0`, do the deleted rows come back?
4. What happens if you query `TIMESTAMP AS OF` a timestamp *before* the table's first version?

In [0]:
# E4: UPDATE and DELETE, then time-travel back (VERSION AS OF and TIMESTAMP AS OF)
# CTAS copies the encounters table (its first version).
spark.sql("""
CREATE OR REPLACE TABLE main.dbacademy_l01.ex_enc_demo AS
SELECT encounter_id, patient_id, encounter_type, diagnosis_code
FROM main.dbacademy_l01.encounters
""")
display(spark.sql("SELECT COUNT(*) AS v0_count FROM main.dbacademy_l01.ex_enc_demo"))
display(spark.sql("SELECT COUNT(*) AS telehealth_before FROM main.dbacademy_l01.ex_enc_demo WHERE encounter_type = 'TELEHEALTH'"))

# UPDATE: rename TELEHEALTH -> VIDEO_VISIT (the next version)
spark.sql("UPDATE main.dbacademy_l01.ex_enc_demo SET encounter_type = 'VIDEO_VISIT' WHERE encounter_type = 'TELEHEALTH'")
display(spark.sql("SELECT COUNT(*) AS telehealth_after_update FROM main.dbacademy_l01.ex_enc_demo WHERE encounter_type = 'TELEHEALTH'"))
display(spark.sql("SELECT COUNT(*) AS video_visit_after_update FROM main.dbacademy_l01.ex_enc_demo WHERE encounter_type = 'VIDEO_VISIT'"))

# DELETE: remove EMERGENCY rows (the next version)
spark.sql("DELETE FROM main.dbacademy_l01.ex_enc_demo WHERE encounter_type = 'EMERGENCY'")
display(spark.sql("SELECT COUNT(*) AS emergency_after_delete FROM main.dbacademy_l01.ex_enc_demo WHERE encounter_type = 'EMERGENCY'"))
display(spark.sql("SELECT COUNT(*) AS total_after_delete FROM main.dbacademy_l01.ex_enc_demo"))

# Time travel by VERSION: read the table as it was at version 0 (before UPDATE and DELETE)
display(spark.sql("SELECT COUNT(*) AS tt_v0_count FROM main.dbacademy_l01.ex_enc_demo VERSION AS OF 0"))
display(spark.sql("SELECT COUNT(*) AS tt_telehealth_v0 FROM main.dbacademy_l01.ex_enc_demo VERSION AS OF 0 WHERE encounter_type = 'TELEHEALTH'"))
display(spark.sql("SELECT COUNT(*) AS tt_emergency_v0 FROM main.dbacademy_l01.ex_enc_demo VERSION AS OF 0 WHERE encounter_type = 'EMERGENCY'"))

# Time travel by TIMESTAMP: a timestamp BEFORE the first version is an error
print("=== TIMESTAMP AS OF before the first version (expected to fail) ===")
try:
    display(spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_enc_demo TIMESTAMP AS OF '2020-01-01 00:00:00'"))
except Exception as e:
    print("failed:", str(e).splitlines()[0])

# A working TIMESTAMP AS OF uses a real commit timestamp from DESCRIBE HISTORY
print("=== TIMESTAMP AS OF using a commit timestamp from DESCRIBE HISTORY ===")
hist_ts = spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_enc_demo").select("version","timestamp").orderBy("version").collect()
v1_ts = hist_ts[1]["timestamp"].strftime("%Y-%m-%d %H:%M:%S")   # the version-1 commit timestamp
print("using timestamp of version 1:", v1_ts)
display(spark.sql(f"SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_enc_demo TIMESTAMP AS OF '{v1_ts}'"))

display(spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_enc_demo"))  # audit trail of every operation

**Explained (real output from this run):**

* Version 0 (CTAS): **300** rows; **81** were `TELEHEALTH`.
* After `UPDATE`: `TELEHEALTH` = **0**, `VIDEO_VISIT` = **81** (all 81 renamed).
* After `DELETE` of `EMERGENCY`: `EMERGENCY` = **0**, total = **224** (= 300 − 76 deleted).
* Time travel to `VERSION AS OF 0`: count = **300**, `TELEHEALTH` = **81**, `EMERGENCY` = **76**. The old data is still there — deletes/updates did **not** erase history.
* `DESCRIBE HISTORY` showed (in order): `CREATE OR REPLACE TABLE AS SELECT` (v0), `UPDATE` with `numUpdatedRows=81`, a serverless `OPTIMIZE`, `DELETE` with `numDeletedRows=76`, and another `OPTIMIZE`.

**Time travel by TIMESTAMP (verified in this run):**

* `TIMESTAMP AS OF '2020-01-01'` (a timestamp *before* the table's first version) failed with `[DELTA_TIMESTAMP_EARLIER_THAN_COMMIT_RETENTION]` — the timestamp is earlier than the earliest available version.
* `TIMESTAMP AS OF <version-1 commit timestamp>` (taken from `DESCRIBE HISTORY`) returned **300** rows — the table state right after the `UPDATE` (same as `VERSION AS OF 1`).

> **Exam point:** `TIMESTAMP AS OF` only works for timestamps within the table's retained history. Use a real commit timestamp from `DESCRIBE HISTORY`, not an arbitrary past date. `VERSION AS OF n` is safer because version numbers are exact.
>
> **Verified behavior:** `UPDATE`/`DELETE` use **deletion vectors** (you see `numDeletionVectorsAdded=1`), so the physical rows are marked removed rather than rewritten immediately. The auto-`OPTIMIZE` versions compact the files afterward.

### Example E5 — MERGE INTO with all three WHEN branches

**Predict:**
1. With 8 target rows and 5 source rows, how many rows survive the `WHEN NOT MATCHED BY SOURCE THEN DELETE`?
2. Which branch handles rows that exist in the source but not the target — insert or delete?
3. Which `operationMetrics` fields report inserted/updated/deleted counts?

In [0]:
# E5: MERGE using all three branches: UPDATE matched, INSERT not-matched, DELETE not-matched-by-source
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_merge_target")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_merge_source")

spark.sql("CREATE TABLE main.dbacademy_l01.ex_merge_target (claim_id INT, claim_status STRING, claim_amount DOUBLE)")
spark.sql("INSERT INTO main.dbacademy_l01.ex_merge_target VALUES "
          "(1,'PENDING',100),(2,'PENDING',200),(3,'PENDING',300),"
          "(4,'APPROVED',400),(5,'APPROVED',500),(6,'APPROVED',600),"
          "(7,'DENIED',700),(8,'DENIED',800)")   # 8 target rows (ids 1-8)

spark.sql("CREATE TABLE main.dbacademy_l01.ex_merge_source (claim_id INT, claim_status STRING, claim_amount DOUBLE)")
spark.sql("INSERT INTO main.dbacademy_l01.ex_merge_source VALUES "
          "(1,'APPROVED',110),(2,'APPROVED',220),(3,'APPROVED',330),"   # match ids 1,2,3 -> UPDATE
          "(9,'PENDING',900),(10,'PENDING',1000)")                       # no match -> INSERT; ids 4-8 absent -> DELETE

display(spark.sql("SELECT COUNT(*) AS target_before FROM main.dbacademy_l01.ex_merge_target"))

spark.sql("""
MERGE INTO main.dbacademy_l01.ex_merge_target AS t
USING main.dbacademy_l01.ex_merge_source  AS s
ON t.claim_id = s.claim_id
WHEN MATCHED THEN UPDATE SET t.claim_status = s.claim_status, t.claim_amount = s.claim_amount
WHEN NOT MATCHED THEN INSERT (claim_id, claim_status, claim_amount) VALUES (s.claim_id, s.claim_status, s.claim_amount)
WHEN NOT MATCHED BY SOURCE THEN DELETE
""")

display(spark.sql("SELECT * FROM main.dbacademy_l01.ex_merge_target ORDER BY claim_id"))
display(spark.sql("SELECT COUNT(*) AS target_after FROM main.dbacademy_l01.ex_merge_target"))
display(spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_merge_target"))  # read operationMetrics

**Explained (real output from this run):**

* Target before: **8** rows. Target after: **5** rows.
* Final rows: ids **1,2,3** (now `APPROVED` — updated) and **9,10** (`PENDING` — inserted). Ids 4–8 were deleted because they had no source match.
* `DESCRIBE HISTORY` MERGE `operationMetrics`:
  * `numTargetRowsInserted` = **2**
  * `numTargetRowsUpdated` = **3**
  * `numTargetRowsDeleted` = **5** (also `numTargetRowsNotMatchedBySourceDeleted` = **5**)
  * `numSourceRows` = **5**, `numOutputRows` = **5**

> **Key takeaway:** `WHEN MATCHED` updates 3 rows, `WHEN NOT MATCHED` inserts 2 rows, `WHEN NOT MATCHED BY SOURCE` deletes 5 rows. The metrics confirm each branch's effect.

### Example E6 — The same MERGE in PySpark (DeltaTable API)

**Predict:**
1. What does `DeltaTable.forName(spark, "table")` return — a DataFrame or a Delta table handle?
2. In `.merge(condition, ...)`, is the join condition a Column or a SQL string?
3. What operation name appears in history for `saveAsTable` with `mode("overwrite")`?

In [0]:
# E6: same MERGE as E5, but in PySpark using the DeltaTable API
from delta.tables import DeltaTable

spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_merge_py_target")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_merge_py_source")

# Build the target as a DataFrame and save it as a managed Delta table
target_data = [(1,'PENDING',100),(2,'PENDING',200),(3,'PENDING',300),
               (4,'APPROVED',400),(5,'APPROVED',500),(6,'APPROVED',600),
               (7,'DENIED',700),(8,'DENIED',800)]
spark.createDataFrame(target_data, ["claim_id","claim_status","claim_amount"]) \
     .write.mode("overwrite").saveAsTable("main.dbacademy_l01.ex_merge_py_target")

# Build the source the same way
source_data = [(1,'APPROVED',110),(2,'APPROVED',220),(3,'APPROVED',330),
               (9,'PENDING',900),(10,'PENDING',1000)]
spark.createDataFrame(source_data, ["claim_id","claim_status","claim_amount"]) \
     .write.mode("overwrite").saveAsTable("main.dbacademy_l01.ex_merge_py_source")

before = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_merge_py_target").collect()[0]["c"]
print("target rows before MERGE:", before)

# Get a Delta table handle (not a plain DataFrame) and the source as a DataFrame
dt  = DeltaTable.forName(spark, "main.dbacademy_l01.ex_merge_py_target")  # Delta table handle
src = spark.table("main.dbacademy_l01.ex_merge_py_source")              # source DataFrame

# Chain the three branches, then execute
(dt.alias("t")
   .merge(src.alias("s"), "t.claim_id = s.claim_id")                    # join condition as a SQL string
   .whenMatchedUpdate(set={"claim_status": "s.claim_status", "claim_amount": "s.claim_amount"})
   .whenNotMatchedInsert(values={"claim_id": "s.claim_id", "claim_status": "s.claim_status", "claim_amount": "s.claim_amount"})
   .whenNotMatchedBySourceDelete()
   .execute())

after = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_merge_py_target").collect()[0]["c"]
print("target rows after MERGE:", after)
spark.sql("SELECT * FROM main.dbacademy_l01.ex_merge_py_target ORDER BY claim_id").show()
print("=== DESCRIBE HISTORY ===")
spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_merge_py_target").select("version","operation","operationMetrics").show(truncate=False)

**Explained (real output from this run):**

* Target before: **8**, after: **5** — identical result to E5.
* `DESCRIBE HISTORY` MERGE `operationMetrics`: `numTargetRowsInserted=2`, `numTargetRowsUpdated=3`, `numTargetRowsDeleted=5`.
* The `saveAsTable(... mode("overwrite"))` step appeared in history as operation **`CREATE OR REPLACE TABLE AS SELECT`** (version 0), `numOutputRows=8`.

> **Exam point:** `DeltaTable.forName(spark, name)` returns a **Delta table handle**, not a DataFrame. The `.merge()` condition can be a SQL string (`"t.id = s.id"`) or a Column expression. `set`/`values` maps use **column-name strings to SQL-string expressions** (e.g. `"s.claim_status"`).
>
> **Minor detail (verified):** because the integer literals had no decimals, `claim_amount` here is `long`; in E5 it was `DOUBLE`. The merge logic is unaffected.

### Example E7 — RESTORE TABLE to an earlier version

**Predict:**
1. After `RESTORE TABLE ... TO VERSION AS OF 0`, what is the row count of `ex_enc_demo` (which had 224 rows after E4)?
2. Does `RESTORE` erase the history, or add a new version?
3. Are the UPDATE and DELETE from E4 reverted by the restore?

> This is a **Python** cell because serverless predictive optimization can run an `OPTIMIZE` at the same moment as `RESTORE`, causing a `ConcurrentWriteException`. We add a small retry loop so the restore is reliable.

In [0]:
# E7: RESTORE the ex_enc_demo table (from E4) back to version 0
import time

before = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_enc_demo").collect()[0]["c"]
print("rows before RESTORE:", before)

latest = spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_enc_demo") \
    .select("version").orderBy("version", ascending=False).first()["version"]
print("latest version before RESTORE:", latest)

# RESTORE syntax: TO VERSION AS OF  (writing "TO VERSION 0" is a syntax error).
# Retry a few times in case serverless auto-OPTIMIZE races with the RESTORE.
def restore_with_retry(table, version, retries=6):
    for i in range(retries):
        try:
            spark.sql(f"RESTORE TABLE {table} TO VERSION AS OF {version}")
            return True
        except Exception as e:
            if "ConcurrentWrite" in str(e):   # auto-OPTIMIZE conflicted; wait and retry
                time.sleep(3); continue
            raise
    return False

ok = restore_with_retry("main.dbacademy_l01.ex_enc_demo", 0)
print("RESTORE succeeded:", ok)

after = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_enc_demo").collect()[0]["c"]
print("rows after RESTORE TO VERSION AS OF 0:", after)
tele = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_enc_demo WHERE encounter_type='TELEHEALTH'").collect()[0]["c"]
emerg = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_enc_demo WHERE encounter_type='EMERGENCY'").collect()[0]["c"]
print("TELEHEALTH after restore:", tele, "(UPDATE reverted)")
print("EMERGENCY after restore:", emerg, "(DELETE reverted)")

print("=== DESCRIBE HISTORY after RESTORE ===")
spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_enc_demo") \
    .select("version","operation","operationMetrics").show(truncate=False)

**Explained (real output from this run):**

* Before RESTORE: **224** rows; latest version was **4** (after CTAS, UPDATE, OPTIMIZE, DELETE, OPTIMIZE).
* After `RESTORE TABLE ... TO VERSION AS OF 0`: **300** rows; `TELEHEALTH` = **81**, `EMERGENCY` = **76**. The UPDATE and DELETE were reverted.
* `DESCRIBE HISTORY` now shows a new version with operation **`RESTORE`** — it did **not** erase the old versions. Its metrics included `numRestoredFiles=1` and `numOfFilesAfterRestore=1`.

> **Key takeaway:** `RESTORE` adds a new version; it does not truncate history. The correct syntax is `TO VERSION AS OF n`. On serverless, RESTORE can race with auto-OPTIMIZE, so a retry loop is a realistic safeguard.

### Example E8 — Temp view vs global temp view vs persistent table

**Predict:**
1. Where does a session temp view appear — in `SHOW TABLES`, `SHOW VIEWS`, or both?
2. Can you query a global temp view as `SELECT * FROM tv_global` (no prefix)?
3. On serverless compute, does `createGlobalTempView` succeed?

In [0]:
# E8: session temp view, persistent UC table, and a global temp view attempt
spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA dbacademy_l01")

# 1) Session-scoped temp view (lives only in this Spark session)
spark.sql("SELECT claim_id, claim_status FROM claims WHERE claim_status='APPROVED' LIMIT 3") \
     .createOrReplaceTempView("tv_session")
print("=== session temp view ===")
spark.sql("SELECT * FROM tv_session").show()

print("=== SHOW VIEWS (temp views show here) ===")
spark.sql("SHOW VIEWS").show(truncate=False)

# 2) Persistent UC table (governed, survives restarts)
print("=== persistent table ===")
print("claims count:", spark.sql("SELECT COUNT(*) AS c FROM claims").collect()[0]["c"])

# 3) Global temp view attempt — capture the real error on serverless
print("=== global temp view attempt on serverless ===")
try:
    spark.sql("SELECT claim_id FROM claims LIMIT 1").createGlobalTempView("tv_global")
    print("unexpectedly created")
except Exception as e:
    print("failed as expected:", str(e).splitlines()[0])

print("=== SHOW TABLES (temp view appears with isTemporary=true) ===")
spark.sql("SHOW TABLES").show(truncate=False)

**Explained (real output from this run):**

* Session temp view `tv_session` returned **3** `APPROVED` claims (the exact ids depend on the `LIMIT 3` order, which is not deterministic without `ORDER BY`).
* `SHOW VIEWS` listed `tv_session` with `isTemporary=true` and an empty namespace.
* Persistent table `claims` count = **200**.
* `createGlobalTempView` **failed** with `[NOT_SUPPORTED_WITH_SERVERLESS] GLOBAL TEMPORARY VIEW is not supported on serverless compute.`
* `SHOW TABLES` also listed `tv_session` with `isTemporary=true` and an empty `database` column — temp views are not UC tables.

> **Verified:** global temp views are **not supported on serverless compute**. Use a session temp view (`createOrReplaceTempView`) or a persistent UC table/view instead. On classic compute, a global temp view is queried via the `global_temp.` schema prefix.

### Example E9 — INSERT OVERWRITE vs CREATE OR REPLACE vs DROP+CREATE vs UNDROP (history)

**Predict:**
1. After `INSERT OVERWRITE`, can you still time-travel to the rows from before the overwrite?
2. Does `CREATE OR REPLACE TABLE` reset the version counter back to 0, or continue it?
3. After `DROP TABLE`, is the table gone forever? What does `DROP TABLE` + `CREATE TABLE` do to the version counter, and can a dropped table be recovered?

In [0]:
# E9: compare INSERT OVERWRITE, CREATE OR REPLACE, DROP+CREATE, and UNDROP
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_overwrite")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_cor")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_dropcreate")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_undrop")

# A) INSERT OVERWRITE: history CONTINUES (old versions stay readable)
spark.sql("CREATE TABLE main.dbacademy_l01.ex_overwrite (id INT, val STRING)")          # first version
spark.sql("INSERT INTO main.dbacademy_l01.ex_overwrite VALUES (1,'a'),(2,'b')")        # next version (2 rows)
spark.sql("INSERT OVERWRITE main.dbacademy_l01.ex_overwrite VALUES (10,'x'),(20,'y'),(30,'z')")  # next version (3 rows)
display(spark.sql("SELECT COUNT(*) AS overwrite_rows FROM main.dbacademy_l01.ex_overwrite"))
display(spark.sql("SELECT COUNT(*) AS overwrite_tt_v1 FROM main.dbacademy_l01.ex_overwrite VERSION AS OF 1"))  # old 2 rows still there
display(spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_overwrite"))

# B) CREATE OR REPLACE TABLE: history CONTINUES (does NOT reset)
spark.sql("CREATE TABLE main.dbacademy_l01.ex_cor (id INT, val STRING)")               # first version
spark.sql("INSERT INTO main.dbacademy_l01.ex_cor VALUES (1,'a'),(2,'b')")              # next version
spark.sql("CREATE OR REPLACE TABLE main.dbacademy_l01.ex_cor (id INT, val STRING)")   # new empty table, history kept
spark.sql("INSERT INTO main.dbacademy_l01.ex_cor VALUES (10,'x'),(20,'y')")            # next version
display(spark.sql("SELECT COUNT(*) AS cor_rows FROM main.dbacademy_l01.ex_cor"))
display(spark.sql("SELECT COUNT(*) AS cor_tt_v1 FROM main.dbacademy_l01.ex_cor VERSION AS OF 1"))  # old data still readable!
display(spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_cor"))

# C) DROP + CREATE: history RESETS (brand-new table, old versions are gone)
spark.sql("CREATE TABLE main.dbacademy_l01.ex_dropcreate (id INT, val STRING)")
spark.sql("INSERT INTO main.dbacademy_l01.ex_dropcreate VALUES (1,'a'),(2,'b')")
spark.sql("DROP TABLE main.dbacademy_l01.ex_dropcreate")                              # table + history removed
spark.sql("CREATE TABLE main.dbacademy_l01.ex_dropcreate (id INT, val STRING)")       # brand-new first version
spark.sql("INSERT INTO main.dbacademy_l01.ex_dropcreate VALUES (10,'x'),(20,'y')")    # next version
display(spark.sql("SELECT COUNT(*) AS dropcreate_rows FROM main.dbacademy_l01.ex_dropcreate"))
display(spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_dropcreate"))  # only 2 versions: the NEW table

# D) UNDROP: a dropped UC managed table can be brought back WITH its history (within the retention window)
spark.sql("CREATE TABLE main.dbacademy_l01.ex_undrop (id INT, val STRING)")
spark.sql("INSERT INTO main.dbacademy_l01.ex_undrop VALUES (1,'a'),(2,'b')")
display(spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_undrop"))   # 2 versions before the drop
spark.sql("DROP TABLE main.dbacademy_l01.ex_undrop")                  # dropped -> not queryable
print("after DROP, SELECT fails:")
try:
    spark.sql("SELECT COUNT(*) FROM main.dbacademy_l01.ex_undrop").show()
except Exception as e:
    print("  ", str(e).splitlines()[0])
spark.sql("UNDROP TABLE main.dbacademy_l01.ex_undrop")                # bring the table back
display(spark.sql("SELECT COUNT(*) AS undrop_rows FROM main.dbacademy_l01.ex_undrop"))  # rows are back
display(spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_undrop"))  # history is restored

# DESCRIBE DETAIL shows physical table info (format, file count, size, location)
display(spark.sql("DESCRIBE DETAIL main.dbacademy_l01.ex_overwrite"))

**Explained (real output from this run):**

| Operation | Final rows | Versions in history | Old data still readable? |
| --- | --- | --- | --- |
| **INSERT OVERWRITE** | 3 | v0 `CREATE TABLE`, v1 `WRITE`(2), v2 `WRITE`(3) | **Yes** — `VERSION AS OF 1` = 2 rows |
| **CREATE OR REPLACE TABLE** | 2 | v0 `CREATE TABLE`, v1 `WRITE`, v2 `CREATE OR REPLACE TABLE`, v3 `WRITE` | **Yes** — `VERSION AS OF 1` = 2 rows |
| **DROP + CREATE** | 2 | v0 `CREATE TABLE`, v1 `WRITE` (only 2) | **No** — old table's history is gone |
| **DROP + UNDROP** | 2 | v0 `CREATE TABLE`, v1 `WRITE` (restored) | **Yes** — `UNDROP TABLE` brings the dropped table back with its history |

* `DESCRIBE DETAIL` on `ex_overwrite` returned `format=delta`, `numFiles=1`, `sizeInBytes=1014`, plus `id`, `name`, `location`, `createdAt`, `partitionColumns`, etc.

> **Most important finding (verified):** `CREATE OR REPLACE TABLE` does **not** reset history — it adds a new version and old versions stay time-travelable. Only `DROP TABLE` + `CREATE TABLE` gives you a brand-new table with a fresh version 0 — but a dropped UC managed table can be recovered with `UNDROP TABLE` within the retention window (history and data come back). Many people assume `CREATE OR REPLACE` resets versions — it does not.

## 5 Exam Traps

Every row below was verified (or explicitly marked concept-only) in this notebook.

| # | Trap (the wrong belief) | Truth (verified) |
| --- | --- | --- |
| 1 | "The default catalog is always `main`." | The workspace default can be something else — here it was `bunyod`. Always `USE CATALOG` explicitly. |
| 2 | "`current_schema()` returns `catalog.schema`." | It returns only the schema name (e.g. `dbacademy_l01`). Pair it with `current_catalog()`. |
| 3 | "`CREATE OR REPLACE TABLE` resets the version counter to 0." | It does **not**. It adds a new version and keeps old versions time-travelable (E9). Only `DROP`+`CREATE` gives a fresh version 0 — but the dropped table can be recovered with `UNDROP TABLE` within the retention window (E9). |
| 4 | "`INSERT OVERWRITE` deletes old history." | It appends a new `WRITE` version; old versions stay readable (E9). |
| 5 | "`RESTORE` syntax is `RESTORE TABLE t TO VERSION 0`." | The syntax is `RESTORE TABLE t TO VERSION AS OF 0` (needs `AS OF`). Without it: syntax error (E7). |
| 6 | "`RESTORE` erases the history after the target version." | `RESTORE` adds a **new** `RESTORE` version; all old versions remain readable (E7). |
| 7 | "`WHEN NOT MATCHED` deletes target rows." | No — it **inserts** source rows. `WHEN NOT MATCHED BY SOURCE` is what acts on target rows (delete/update) (E5). |
| 8 | "Every version in `DESCRIBE HISTORY` is one I wrote." | Serverless **predictive optimization** adds `OPTIMIZE` versions you did not write (E4, E5). Read the `operation` column. |
| 9 | "Global temp views work everywhere." | They are **not supported on serverless compute** (`NOT_SUPPORTED_WITH_SERVERLESS`). Use a session temp view or a UC table (E8). |
| 10 | "A session temp view shows up in `SHOW TABLES`." | It does, with `isTemporary=true` and an empty `database` — but it is not a governed UC table (E8). |
| 11 | "`saveAsTable(mode="overwrite")` shows as `WRITE` in history." | It shows as `CREATE OR REPLACE TABLE AS SELECT` (E6). |
| 12 | "`UPDATE`/`DELETE` rewrite all files immediately." | They use **deletion vectors** (`numDeletionVectorsAdded`); files are compacted later by a background `OPTIMIZE` (E4). |
| 13 | "`DROP TABLE` permanently destroys a managed table." | A dropped UC managed table can be recovered with `UNDROP TABLE` within the retention window — history and data come back (E9). |

## 6 Syntax Cheat Sheet (SQL ↔ PySpark)

### DML side by side

| Goal | SQL | PySpark |
| --- | --- | --- |
| Create table from a query | `CREATE [OR REPLACE] TABLE t AS SELECT ...` | `df.write.mode("overwrite").saveAsTable("t")` |
| Append rows | `INSERT INTO t VALUES ...` | `df.write.mode("append").saveAsTable("t")` |
| Overwrite all rows | `INSERT OVERWRITE t SELECT ...` | `df.write.mode("overwrite").saveAsTable("t")` |
| Update rows | `UPDATE t SET col=val WHERE ...` | `DeltaTable.forName(spark,"t").update(condition, {"col":"val"})` |
| Delete rows | `DELETE FROM t WHERE ...` | `DeltaTable.forName(spark,"t").delete("condition")` |
| Upsert | `MERGE INTO t USING s ON ... WHEN ...` | `DeltaTable.forName(spark,"t").merge(df,"cond").whenMatchedUpdate(...).whenNotMatchedInsert(...).execute()` |

### History & time travel

| Goal | Syntax |
| --- | --- |
| Audit trail | `DESCRIBE HISTORY t` |
| Physical info | `DESCRIBE DETAIL t` |
| Schema + extended | `DESCRIBE EXTENDED t` |
| Read old version | `SELECT ... FROM t VERSION AS OF n` |
| Read at timestamp | `SELECT ... FROM t TIMESTAMP AS OF '2024-06-01T00:00:00'` |
| Roll back | `RESTORE TABLE t TO VERSION AS OF n` |

### Navigation

| Goal | Syntax |
| --- | --- |
| Set default catalog | `USE CATALOG main` |
| Set default schema | `USE SCHEMA dbacademy_l01` |
| List catalogs / schemas / tables | `SHOW CATALOGS` / `SHOW SCHEMAS IN main` / `SHOW TABLES` |
| Check defaults | `SELECT current_catalog(), current_schema()` |

### MERGE branch quick reference

| Branch | Acts on | Default action |
| --- | --- | --- |
| `WHEN MATCHED` | target row that joins a source row | UPDATE or DELETE |
| `WHEN NOT MATCHED` | source row with no target match | INSERT |
| `WHEN NOT MATCHED BY SOURCE` | target row with no source match | DELETE or UPDATE |

> **MERGE clause rules (verified against Databricks docs):** there can be **any number** of `WHEN MATCHED`, `WHEN NOT MATCHED`, and `WHEN NOT MATCHED BY SOURCE` clauses. Multiple clauses of the same kind are evaluated **in order**; **all except the last must have a condition**. A `WHEN MATCHED` clause can do at most one action (either UPDATE or DELETE).

## 7 Check Your Understanding

Answer these in your own words (answers in the Solutions section).

1. You run `SELECT * FROM t VERSION AS OF 5` and then `RESTORE TABLE t TO VERSION AS OF 5`. Both reference version 5. What is the **difference** in what they do to the table and to its history?
2. A teammate says: "I'll use `CREATE OR REPLACE TABLE` to reset the table's history before each load." Why is that statement wrong, and what would actually reset the history?
3. You `MERGE` a daily claims file into a target table with `WHEN MATCHED THEN UPDATE` and `WHEN NOT MATCHED THEN INSERT`, but you forget `WHEN NOT MATCHED BY SOURCE`. What happens to target rows that are no longer in the daily file?

## 8 Practice (4 levels)

Do these yourself **without AI help**. Each practice cell has a validation cell right after it. The solutions are in section 10 — do not peek.

> The practice code cells are intentionally blank (`<FILL_IN>` / `TODO`). Fill them in, then run the validation cell. If a table is missing, the validation prints a hint instead of crashing.

### P1 — Fill in the blanks (3 blanks): CTAS + INSERT + DESCRIBE HISTORY

Create a table of `APPROVED` claims, insert one new row, and read the history. Replace the three `<FILL_IN>` markers.

In [0]:
# P1: Fill in the 3 blanks (<FILL_IN>). Run the validation cell after.
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_pr1_claims")

# Blank 1 & 2: CTAS — pick the right columns and the right status filter
spark.sql("""
CREATE OR REPLACE TABLE main.dbacademy_l01.ex_pr1_claims AS
SELECT <FILL_IN: column list>
FROM main.dbacademy_l01.claims
WHERE claim_status = <FILL_IN: the APPROVED literal>
""")

# Blank 3: INSERT one new row: claim_id=99001, patient_id=1, claim_amount=999.00, claim_status='APPROVED'
spark.sql("""
INSERT INTO main.dbacademy_l01.ex_pr1_claims
VALUES <FILL_IN: one row tuple>
""")

print("rows:", spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr1_claims").collect()[0]["c"])
spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_pr1_claims").select("version","operation").show()

In [0]:
# P1 validation — run AFTER filling in the blanks above
try:
    cnt = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr1_claims").collect()[0]["c"]
    has_new = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr1_claims WHERE claim_id=99001").collect()[0]["c"]
    hv = spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_pr1_claims").select("version","operation").collect()
    if cnt == 54 and has_new == 1 and len(hv) >= 2:
        print(f"✅ Passed — P1 complete ({cnt} rows, claim 99001 inserted, {len(hv)} versions)")
    else:
        print(f"Hint: expected 54 rows, got {cnt}; claim 99001 present={has_new}; versions={len(hv)} (want >=2). Check the column list, the status literal, and the VALUES tuple.")
except Exception as e:
    print("Hint: table not created yet — fill in the blanks in the P1 code cell first.", str(e).splitlines()[0])

### P2 — Fix the bug (2 bugs): a MERGE with a wrong join condition and a missing branch

The setup creates the target and source for you. The MERGE has **two bugs** — fix them in place, then run the validation. Expected result: 4 rows (claim 1 updated to amount 150.0; claims 2, 3 unchanged; claim 4 inserted).

In [0]:
# P2: Fix the 2 bugs in this MERGE (edit this cell in place).
#   BUG 1: the join condition is on the wrong column.
#   BUG 2: a branch is missing, so new source rows are never inserted.
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_pr2_target")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_pr2_source")
spark.sql("CREATE TABLE main.dbacademy_l01.ex_pr2_target (claim_id INT, patient_id INT, claim_status STRING, claim_amount DOUBLE)")
spark.sql("INSERT INTO main.dbacademy_l01.ex_pr2_target VALUES (1,10,'PENDING',100.0),(2,20,'PENDING',200.0),(3,30,'PENDING',300.0)")
spark.sql("CREATE TABLE main.dbacademy_l01.ex_pr2_source (claim_id INT, patient_id INT, claim_status STRING, claim_amount DOUBLE)")
spark.sql("INSERT INTO main.dbacademy_l01.ex_pr2_source VALUES (1,10,'APPROVED',150.0),(4,40,'PENDING',400.0)")

# Fix the two bugs below:
spark.sql("""
MERGE INTO main.dbacademy_l01.ex_pr2_target AS t
USING main.dbacademy_l01.ex_pr2_source AS s
ON t.patient_id = s.patient_id   -- BUG 1: wrong join column
WHEN MATCHED THEN UPDATE SET t.claim_status = s.claim_status, t.claim_amount = s.claim_amount
                                 -- BUG 2: missing branch for new source rows
""")
spark.sql("SELECT * FROM main.dbacademy_l01.ex_pr2_target ORDER BY claim_id").show()

In [0]:
# P2 validation
try:
    cnt = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr2_target").collect()[0]["c"]
    has4 = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr2_target WHERE claim_id=4").collect()[0]["c"]
    amt1 = spark.sql("SELECT claim_amount FROM main.dbacademy_l01.ex_pr2_target WHERE claim_id=1").collect()[0]["claim_amount"]
    if cnt == 4 and has4 == 1 and amt1 == 150.0:
        print(f"✅ Passed — P2 fixed (4 rows, claim 4 inserted, claim 1 amount = {amt1})")
    else:
        print(f"Hint: expected 4 rows got {cnt}; claim 4 present={has4}; claim 1 amount={amt1} (want 150.0). Fix the join key and add the missing insert branch.")
except Exception as e:
    print("Hint: run the P2 code cell first.", str(e).splitlines()[0])

### P3 — From scratch (with hints): daily claims upsert with MERGE + time travel (SQL)

Implement a daily upsert in SQL, then time-travel to compare before/after.

**Hints:**
* Target `main.dbacademy_l01.ex_pr3_claims`: CTAS of `claims` where `claim_id <= 20` (columns `claim_id, patient_id, claim_status, claim_amount`).
* Source `main.dbacademy_l01.ex_pr3_stg`: claims 1–5 with `claim_status='APPROVED'`, plus claims 21, 22, 23 with `claim_status='PENDING'`.
* `MERGE`: `WHEN MATCHED` update status; `WHEN NOT MATCHED` insert.
* After the merge, query `VERSION AS OF 0` and print its count (should be 20).

Expected: 20 rows before, 23 after; claims 1–5 become `APPROVED`; claims 21–23 inserted.

In [0]:
# P3: write your SQL upsert here. Replace the TODO lines with real code.
# TODO 1: CTAS the target (claims 1-20)
# TODO 2: build the source staging table (claims 1-5 -> APPROVED; claims 21,22,23 -> PENDING)
# TODO 3: MERGE target USING source: WHEN MATCHED UPDATE status; WHEN NOT MATCHED INSERT
# TODO 4: print the count at VERSION AS OF 0 (should be 20)

# Your code here


In [0]:
# P3 validation
try:
    before = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr3_claims VERSION AS OF 0").collect()[0]["c"]
    after = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr3_claims").collect()[0]["c"]
    appr = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr3_claims WHERE claim_id <= 5 AND claim_status='APPROVED'").collect()[0]["c"]
    newc = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr3_claims WHERE claim_id IN (21,22,23)").collect()[0]["c"]
    if before == 20 and after == 23 and appr == 5 and newc == 3:
        print(f"✅ Passed — P3 upsert works (20 before, {after} after, {appr} updated, {newc} inserted; time travel v0 = {before})")
    else:
        print(f"Hint: before(={before} want 20) after(={after} want 23) approved1-5(={appr} want 5) new21-23(={newc} want 3)")
except Exception as e:
    print("Hint: write and run the P3 code cell first.", str(e).splitlines()[0])

### P4 — BLIND (no hints): P3 again in PySpark + RESTORE

Do P3 again, but this time:
1. Use the **PySpark DeltaTable API** (`DeltaTable.forName(spark, ...).merge(...)`) instead of SQL.
2. Use tables `main.dbacademy_l01.ex_pr4_claims` (target) and `main.dbacademy_l01.ex_pr4_stg` (source), same data shape as P3.
3. After the merge, **`RESTORE`** the table to the version **before** the merge.

The solution goes only in section 10. The validation checks that the merge happened and the restore brought the table back to 20 rows.

In [0]:
# P4: your PySpark solution here (DeltaTable.merge + RESTORE). No hints.
# TODO: create target + source, MERGE with DeltaTable API, then RESTORE to the pre-merge version.

# Your code here


In [0]:
# P4 validation
try:
    v0 = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr4_claims VERSION AS OF 0").collect()[0]["c"]
    restored = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr4_claims").collect()[0]["c"]
    gone = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr4_claims WHERE claim_id IN (21,22,23)").collect()[0]["c"]
    ops = [r["operation"] for r in spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_pr4_claims").select("operation").collect()]
    has_merge = "MERGE" in ops
    has_restore = "RESTORE" in ops
    if v0 == 20 and restored == 20 and gone == 0 and has_merge and has_restore:
        print(f"✅ Passed — P4 complete (v0={v0}, restored={restored}, inserted-now-gone={gone}, MERGE+RESTORE in history)")
    else:
        print(f"Hint: v0(={v0} want 20) restored(={restored} want 20) inserted-gone(={gone} want 0) MERGE={has_merge} RESTORE={has_restore}")
except Exception as e:
    print("Hint: write and run the P4 code cell first.", str(e).splitlines()[0])

## 9 Exam-Style Questions (healthcare context)

Choose the **one best answer** for each. Answers and explanations are in section 10.

**Q1.** A new engineer opens a notebook and runs `SELECT COUNT(*) FROM claims` without any `USE` statement. It fails with "table not found," even though `main.dbacademy_l01.claims` exists. What is the most likely cause?

A. The table is external and has no metadata.
B. The default catalog/schema is not `main.dbacademy_l01`, so the unqualified name does not resolve.
C. The table was created with `CREATE OR REPLACE TABLE`, which hides it from unqualified queries.
D. `claims` is a temp view, and temp views are session-scoped.

**Q2.** You run `CREATE OR REPLACE TABLE main.dbacademy_l01.enc_daily AS SELECT ... FROM encounters` and then `INSERT INTO main.dbacademy_l01.enc_daily VALUES (...)`. How many versions are in `DESCRIBE HISTORY` and what are they?

A. 1 version: `CREATE OR REPLACE TABLE AS SELECT`.
B. 2 versions: `CREATE OR REPLACE TABLE AS SELECT` (v0), then `WRITE` with `mode=Append` (v1).
C. 3 versions: the CTAS resets to v0, plus two `WRITE` operations.
D. 0 versions: `CREATE OR REPLACE` clears history before the insert.

**Q3.** A teammate says: "I'll run `CREATE OR REPLACE TABLE claims_daily ...` at the start of every load so the table's history is reset and old versions are gone." What is the **truth**?

A. Correct — `CREATE OR REPLACE TABLE` resets the version counter to 0.
B. Wrong — `CREATE OR REPLACE TABLE` adds a new version and keeps old versions time-travelable; only `DROP TABLE` + `CREATE TABLE` resets history (and the dropped table can be recovered with `UNDROP TABLE` within the retention window).
C. Wrong — only `INSERT OVERWRITE` can reset history.
D. Correct, but only for external tables.

**Q4.** You `MERGE` a daily claims feed into a target table. Target rows whose `claim_id` no longer appears in the daily feed should be **deleted**. Which `MERGE` branch does this?

A. `WHEN NOT MATCHED THEN DELETE`
B. `WHEN MATCHED AND s.claim_id IS NULL THEN DELETE`
C. `WHEN NOT MATCHED BY SOURCE THEN DELETE`
D. `WHEN NOT MATCHED BY TARGET THEN DELETE`

**Q5.** A bad load corrupted `main.dbacademy_l01.claims` and you want to roll it back to version 12. Which statement is syntactically correct?

A. `RESTORE TABLE main.dbacademy_l01.claims TO VERSION 12`
B. `RESTORE TABLE main.dbacademy_l01.claims VERSION AS OF 12`
C. `RESTORE TABLE main.dbacademy_l01.claims TO VERSION AS OF 12`
D. `RESTORE main.dbacademy_l01.claims TO TIMESTAMP AS OF 12`

**Q6.** A teammate wants to read `main.dbacademy_l01.claims` as it was 3 days ago and writes:
`SELECT * FROM main.dbacademy_l01.claims TIMESTAMP AS OF DATE_SUB(CURRENT_DATE(), 3)`.
The table was created **today** (this lesson's setup). What is the result?

A. It returns 0 rows because the table did not exist 3 days ago.
B. It returns the current rows (a pre-history timestamp is ignored).
C. It fails with `[DELTA_TIMESTAMP_EARLIER_THAN_COMMIT_RETENTION]` — the timestamp is before the earliest available version.
D. It fails with a syntax error — `TIMESTAMP AS OF` requires a version number, not a date expression.

> **Verified in E4:** a working `TIMESTAMP AS OF` uses a real commit timestamp taken from `DESCRIBE HISTORY`, not an arbitrary past date. `VERSION AS OF n` is safer because version numbers are exact.

**Q7.** On a **serverless** notebook, you run `df.createGlobalTempView("gv")` and then `SELECT * FROM global_temp.gv`. What happens?

A. It works; global temp views are visible across all sessions via `global_temp.`.
B. It fails with `NOT_SUPPORTED_WITH_SERVERLESS` because global temp views are not supported on serverless compute.
C. It works, but only inside the current notebook.
D. It works, but the view is deleted when the notebook detaches.

**Q8.** You run only an `UPDATE` and a `DELETE` on a table, but `DESCRIBE HISTORY` shows five versions including two `OPTIMIZE` operations you never wrote. Why?

A. `UPDATE` and `DELETE` each create two versions.
B. Serverless predictive optimization ran background `OPTIMIZE` commits between your writes.
C. The transaction log duplicates every operation.
D. `DESCRIBE HISTORY` shows planned, not executed, operations.

## 10 Solutions

> **Do not scroll here before trying the practice and exam questions yourself.**

### Check Your Understanding — answers

1. `SELECT ... VERSION AS OF 5` is a **read** — it returns the table's data as it was at version 5 without changing anything. `RESTORE TABLE ... TO VERSION AS OF 5` is a **write** — it changes the current table to match version 5 and adds a **new** `RESTORE` version to the history (it does not erase history).
2. `CREATE OR REPLACE TABLE` does **not** reset history — it adds a new version and old versions stay time-travelable (verified in E9). To truly reset history you must `DROP TABLE` and then `CREATE TABLE`, which creates a brand-new table with a fresh version 0 (but the dropped table can be recovered with `UNDROP TABLE` within the retention window).
3. Without `WHEN NOT MATCHED BY SOURCE`, target rows that are absent from the daily file are simply left **unchanged** — they are not deleted. You get an upsert (insert + update) but no removal of stale rows. To also remove rows that left the feed, add `WHEN NOT MATCHED BY SOURCE THEN DELETE`.

### P1 solution

```python
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_pr1_claims")
spark.sql("""
CREATE OR REPLACE TABLE main.dbacademy_l01.ex_pr1_claims AS
SELECT claim_id, patient_id, claim_amount, claim_status
FROM main.dbacademy_l01.claims
WHERE claim_status = 'APPROVED'
""")
spark.sql("""
INSERT INTO main.dbacademy_l01.ex_pr1_claims
VALUES (99001, 1, 999.00, 'APPROVED')
""")
print("rows:", spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr1_claims").collect()[0]["c"])
spark.sql("DESCRIBE HISTORY main.dbacademy_l01.ex_pr1_claims").select("version","operation").show()
# Expected: 54 rows, 2 versions (CTAS = 53 rows, then WRITE append = 1 row)
```

### P2 solution (the two fixes)

```python
spark.sql("""
MERGE INTO main.dbacademy_l01.ex_pr2_target AS t
USING main.dbacademy_l01.ex_pr2_source AS s
ON t.claim_id = s.claim_id                       -- FIX 1: join on claim_id, not patient_id
WHEN MATCHED THEN UPDATE SET t.claim_status = s.claim_status, t.claim_amount = s.claim_amount
WHEN NOT MATCHED THEN INSERT (claim_id, patient_id, claim_status, claim_amount)
     VALUES (s.claim_id, s.patient_id, s.claim_status, s.claim_amount)  -- FIX 2: add the insert branch
""")
# Expected: 4 rows; claim 1 amount = 150.0; claim 4 inserted.
```

### P3 solution

```python
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_pr3_claims")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_pr3_stg")

# 1) target: claims 1-20
spark.sql("""
CREATE OR REPLACE TABLE main.dbacademy_l01.ex_pr3_claims AS
SELECT claim_id, patient_id, claim_status, claim_amount
FROM main.dbacademy_l01.claims WHERE claim_id <= 20
""")

# 2) source: claims 1-5 -> APPROVED, plus claims 21,22,23 -> PENDING
spark.sql("""
CREATE OR REPLACE TABLE main.dbacademy_l01.ex_pr3_stg AS
SELECT claim_id, patient_id, 'APPROVED' AS claim_status, claim_amount
FROM main.dbacademy_l01.claims WHERE claim_id <= 5
""")
spark.sql("""
INSERT INTO main.dbacademy_l01.ex_pr3_stg
SELECT claim_id, patient_id, 'PENDING' AS claim_status, claim_amount
FROM main.dbacademy_l01.claims WHERE claim_id IN (21,22,23)
""")

# 3) MERGE upsert
spark.sql("""
MERGE INTO main.dbacademy_l01.ex_pr3_claims AS t
USING main.dbacademy_l01.ex_pr3_stg AS s
ON t.claim_id = s.claim_id
WHEN MATCHED THEN UPDATE SET t.claim_status = s.claim_status
WHEN NOT MATCHED THEN INSERT (claim_id, patient_id, claim_status, claim_amount)
     VALUES (s.claim_id, s.patient_id, s.claim_status, s.claim_amount)
""")

# 4) time travel to before the merge
print("v0 count (before merge):", spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr3_claims VERSION AS OF 0").collect()[0]["c"])
print("after merge:", spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr3_claims").collect()[0]["c"])
# Expected: v0 = 20, after = 23
```

### P4 solution

```python
import time
from delta.tables import DeltaTable

spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_pr4_claims")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l01.ex_pr4_stg")

# target + source (same shape as P3)
spark.sql("""
CREATE OR REPLACE TABLE main.dbacademy_l01.ex_pr4_claims AS
SELECT claim_id, patient_id, claim_status, claim_amount
FROM main.dbacademy_l01.claims WHERE claim_id <= 20
""")
spark.sql("""
CREATE OR REPLACE TABLE main.dbacademy_l01.ex_pr4_stg AS
SELECT claim_id, patient_id, 'APPROVED' AS claim_status, claim_amount
FROM main.dbacademy_l01.claims WHERE claim_id <= 5
""")
spark.sql("""
INSERT INTO main.dbacademy_l01.ex_pr4_stg
SELECT claim_id, patient_id, 'PENDING' AS claim_status, claim_amount
FROM main.dbacademy_l01.claims WHERE claim_id IN (21,22,23)
""")

# PySpark MERGE via DeltaTable API
dt  = DeltaTable.forName(spark, "main.dbacademy_l01.ex_pr4_claims")
src = spark.table("main.dbacademy_l01.ex_pr4_stg")
(dt.alias("t")
   .merge(src.alias("s"), "t.claim_id = s.claim_id")
   .whenMatchedUpdate(set={"claim_status": "s.claim_status"})
   .whenNotMatchedInsert(values={"claim_id":"s.claim_id","patient_id":"s.patient_id","claim_status":"s.claim_status","claim_amount":"s.claim_amount"})
   .execute())

# RESTORE to the version before the merge (version 0), with a retry for serverless auto-OPTIMIZE races
def restore_with_retry(table, version, retries=6):
    for i in range(retries):
        try:
            spark.sql(f"RESTORE TABLE {table} TO VERSION AS OF {version}")
            return True
        except Exception as e:
            if "ConcurrentWrite" in str(e):
                time.sleep(3); continue
            raise
    return False

restore_with_retry("main.dbacademy_l01.ex_pr4_claims", 0)
print("after restore:", spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l01.ex_pr4_claims").collect()[0]["c"])
# Expected: 20 (merge took it to 23, restore brought it back to 20)
```

### Exam answers

**Q1 → B.** Unqualified names resolve against the current catalog/schema. The workspace default was `bunyod`, not `main.dbacademy_l01`, so `claims` was not found.
* A wrong: external tables have metadata; that is not the issue. * C wrong: `CREATE OR REPLACE` does not hide tables. * D wrong: `claims` is a persistent UC table, not a temp view.

**Q2 → B.** CTAS is version 0 (`CREATE OR REPLACE TABLE AS SELECT`); the append `INSERT` is version 1 (`WRITE`, `mode=Append`).
* A wrong: the insert also creates a version. * C wrong: there are only two operations. * D wrong: `CREATE OR REPLACE` does not clear history.

**Q3 → B.** `CREATE OR REPLACE TABLE` adds a new version and keeps old versions readable (verified in E9). Only `DROP` + `CREATE` resets history (fresh version 0) — but the dropped table can be recovered with `UNDROP TABLE` within the retention window (E9).
* A wrong: it does not reset. * C wrong: `INSERT OVERWRITE` also preserves history. * D wrong: there is no such external-table exception.

**Q4 → C.** `WHEN NOT MATCHED BY SOURCE THEN DELETE` removes target rows that have no match in the source.
* A wrong: `WHEN NOT MATCHED` inserts source rows, it does not delete target rows. * B wrong: there is no `s.claim_id IS NULL` semantics here; matched rows join. * D wrong: `WHEN NOT MATCHED BY TARGET` is not valid syntax.

**Q5 → C.** The correct syntax is `RESTORE TABLE t TO VERSION AS OF 12`.
* A wrong: missing `AS OF` — syntax error (verified in E7). * B wrong: missing `TO`. * D wrong: `12` is a version, not a timestamp, and the keyword order is wrong.

**Q6 → C.** The table was created today, so a timestamp 3 days ago is before the table's first version. `TIMESTAMP AS OF` with such a timestamp fails with `[DELTA_TIMESTAMP_EARLIER_THAN_COMMIT_RETENTION]` (verified in E4). To time-travel by timestamp, use a real commit timestamp from `DESCRIBE HISTORY`.
* A wrong: it does not return 0 rows — it errors. * B wrong: it does not silently return current rows. * D wrong: `TIMESTAMP AS OF` does accept a timestamp/date expression; the failure is a runtime error, not a syntax error.

**Q7 → B.** Global temp views are not supported on serverless compute (`NOT_SUPPORTED_WITH_SERVERLESS`), verified in E8.
* A wrong: that is the classic-compute behavior, not serverless. * C/D wrong: it does not work at all on serverless.

**Q8 → B.** Serverless predictive optimization runs background `OPTIMIZE` commits automatically; these appear in history (verified in E4/E5).
* A wrong: `UPDATE`/`DELETE` each create one version. * C wrong: the log does not duplicate. * D wrong: history shows executed operations.

In [0]:
# 11 Cleanup: drop everything this lesson created
spark.sql("DROP SCHEMA IF EXISTS main.dbacademy_l01 CASCADE")
print("Cleaned up: schema main.dbacademy_l01 dropped (all tables, views, and the volume removed).")

## 12 Final Summary (verified on this run)

### Real key numbers from cells that actually ran

| Example | Key result |
| --- | --- |
| Setup | patients=50, providers=10, encounters=300, claims=200 |
| E1 | default catalog `bunyod`; after `USE CATALOG main` -> `main`; unqualified `patients` count = 50; 7 columns |
| E2 | volume file `providers_sample.csv` = 77 bytes; widget `Cardiology` -> 4 providers |
| E3 | CTAS `ex_denied_claims` = 40 rows; after INSERT = 42; history: v0 CTAS(40), v1 WRITE/Append(2) |
| E4 | 300 rows; UPDATE 81 TELEHEALTH->VIDEO_VISIT; DELETE 76 EMERGENCY -> 224; time travel v0 = 300; TIMESTAMP AS OF before first version errors; working TIMESTAMP AS OF (v1) = 300 |
| E5 | MERGE: inserted=2, updated=3, deleted=5 (NOT MATCHED BY SOURCE); 8 -> 5 rows |
| E6 | PySpark MERGE same result: 8 -> 5; `saveAsTable(overwrite)` = `CREATE OR REPLACE TABLE AS SELECT` |
| E7 | RESTORE TO VERSION AS OF 0: 224 -> 300; new `RESTORE` version added (history kept) |
| E8 | session temp view works; global temp view FAILS `NOT_SUPPORTED_WITH_SERVERLESS` |
| E9 | INSERT OVERWRITE keeps history; CREATE OR REPLACE keeps history (v1 still readable); DROP+CREATE resets to fresh v0; UNDROP restores a dropped table with its history |

### NOT VERIFIED on this serverless compute (concept only)

* **External tables / `LOCATION` / `DROP` file retention** — covered in Lesson 2.
* **Global temp views** — not supported on serverless; classic-compute behavior described conceptually.
* **Classic all-purpose & Jobs compute** (cluster policies, autoscaling, DBR) — not exercised here.

### Things that behaved differently from first expectations

1. **Default catalog was `bunyod`, not `main`.** I assumed `main`; `current_catalog()` proved otherwise. Added a `USE CATALOG main` note throughout.
2. **Serverless predictive optimization inserted `OPTIMIZE` versions** I never wrote (E4, E5). Version numbers are therefore not always contiguous with your own operations — read the `operation` column.
3. **`RESTORE` raced with auto-OPTIMIZE** and raised `ConcurrentWriteException`. Added a retry loop in E7 and P4.
4. **`CREATE OR REPLACE TABLE` does NOT reset history.** I initially expected it to; E9 proved old versions stay time-travelable. Only `DROP`+`CREATE` resets.
5. **RESTORE syntax needs `AS OF`** (`TO VERSION AS OF n`); `TO VERSION n` is a syntax error.
6. **`saveAsTable(mode="overwrite")`** appears in history as `CREATE OR REPLACE TABLE AS SELECT`, not `WRITE`.
7. **Original claims 1-5 already had 2 APPROVED** (ids 1,2), so after a merge-to-APPROVED + restore, the restored count of APPROVED among ids 1-5 is 2, not 0. Practice P4 validation was written to be status-independent (checks that inserted rows 21-23 are gone and count = 20).
8. **`UNDROP TABLE` recovers a dropped UC managed table** with its full history (E9). And `TIMESTAMP AS OF` with a timestamp before the table's first version fails with `DELTA_TIMESTAMP_EARLIER_THAN_COMMIT_RETENTION` (E4) — use a real commit timestamp from `DESCRIBE HISTORY`.
9. **The setup RNG was non-deterministic** because `spark.createDataFrame().write` calls between list generations consumed the global `random` state, shifting the sequence. Fixed by using an isolated `random.Random(42)` instance so all counts (DENIED=40, Cardiology=4, APPROVED=53, etc.) are now reproducible across runs.